# Native NeoOLAF DocRED **dev-only** streaming batch and evaluation — v6.2

This notebook keeps the frozen **v5.1 scientific configuration and evaluator**, but changes the batch orchestration so that:

- only records whose literal JSON key `"type"` equals `"dev"` are selected;
- the JSONL is scanned line by line instead of loaded into memory;
- no list containing all dev documents is constructed;
- at most `DOCUMENT_WORKERS` document jobs are submitted at once;
- the existing `LAYER_WORKERS` parallelism is retained inside each document;
- evaluation is rebuilt by streaming the per-document result files.

The intended workflow is:

1. leave `RUN_ALL_DEV_DOCUMENTS = False` and run the first five `type="dev"` records;
2. inspect the same strict DocRED relation/entity evaluation used previously;
3. change only `RUN_ALL_DEV_DOCUMENTS = True`;
4. rerun the configuration, preflight, and batch cells.

Completed runs under this notebook's batch root are resumed. Gold entities and relations are removed before each native NeoOLAF run and loaded only afterward for evaluation.

## Memory behavior

The parent process keeps only:

- the current JSONL line;
- at most `DOCUMENT_WORKERS` prepared/submitted jobs;
- small aggregate counters for 96 relation types and 13 layers.

Input/gold records and NeoOLAF artifacts are written to disk per document. The complete dev split is never materialized as a Python list or submitted to `ProcessPoolExecutor` all at once.

In [1]:
from __future__ import annotations

import os
import sys
import multiprocessing as mp
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/neoolaf").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the NeoOLAF repository.")


def first_existing_path(label: str, candidates: list[Path]) -> Path:
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser()
        candidate = candidate if candidate.is_absolute() else PROJECT_ROOT / candidate
        candidate = candidate.resolve()
        checked.append(candidate)
        if candidate.is_file():
            print(f"{label}={candidate}")
            return candidate
    raise FileNotFoundError(
        f"Could not find {label}. Checked:\n" + "\n".join(str(path) for path in checked)
    )


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets"
TOOLS_DIR = NOTEBOOK_DIR / "tools"
for path in [PROJECT_ROOT / "src", PROJECT_ROOT, TOOLS_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from docred_native_batch_v6_2_dev_streaming import (
    BatchRunConfig,
    aggregate_batch_results_streaming,
    count_exact_type_records,
    read_json,
    run_batch_streaming,
)

mp.freeze_support()
print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = /home/galencarmedeiro/git/postdoc/NeoOLAF


/home/galencarmedeiro/git/postdoc/NeoOLAF/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
/home/galencarmedeiro/git/postdoc/NeoOLAF/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Exact `type="dev"` filter and one-variable smoke/full switch

In [2]:
# Change only this variable after the five-document dev smoke run succeeds.
RUN_ALL_DEV_DOCUMENTS = True

# Used only while RUN_ALL_DEV_DOCUMENTS=False.
SMOKE_DEV_DOCUMENT_LIMIT = 5

# Index among matching type="dev" records. Keep 0 for the complete dev split.
START_DEV_INDEX = 0

# Frozen exact filter: it checks record["type"] only and does not fall back to "split".
EXACT_TYPE_VALUE = "dev"

DATASET_JSONL = first_existing_path(
    "DATASET_JSONL",
    [
        NOTEBOOK_DIR / "../../../ragtree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT.parent / "ragtree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT.parent / "RAGTree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT / "ragtree/data/preprocessed/docred_causal.jsonl",
    ],
)

# Separate root prevents collisions with the previous unfiltered five-document batch.
# Keep this same root when switching RUN_ALL_DEV_DOCUMENTS to True so smoke runs resume.
BATCH_ROOT = NOTEBOOK_DIR / "runs/docred_native_v5_1_dev_streaming"

print("RUN_ALL_DEV_DOCUMENTS =", RUN_ALL_DEV_DOCUMENTS)
print("DATASET_JSONL =", DATASET_JSONL)
print("BATCH_ROOT =", BATCH_ROOT)

DATASET_JSONL=/home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/docred_causal.jsonl
RUN_ALL_DEV_DOCUMENTS = True
DATASET_JSONL = /home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/docred_causal.jsonl
BATCH_ROOT = /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/docred_native_v5_1_dev_streaming


## 2. Frozen v5.1 scientific configuration and parallel workers

In [3]:
ONTOLOGY_PATH = NOTEBOOK_DIR / "ontology/docred_redocred_neoolaf_compatible.ttl"
ONTOLOGY_ORIGINAL = NOTEBOOK_DIR / "ontology/docred_redocred_original.ttl"
RELATION_CATALOG = NOTEBOOK_DIR / "ontology/docred_relation_catalog.json"
RELATION_ALIASES = NOTEBOOK_DIR / "ontology/docred_relation_aliases.json"
PROFILE_PATH = NOTEBOOK_DIR / "configs/docred_profile_native_ablation_v5.json"
GUIDANCE_PATH = NOTEBOOK_DIR / "configs/guidance_docred_native_ablation_v5.json"
TASK_GUIDANCE_PATH = NOTEBOOK_DIR / "configs/docred_task_guidance_v5_1_frozen.json"

OPENROUTER_HOST = "https://openrouter.ai/api/v1"
MODEL_NAME = "openai/gpt-oss-20b"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")

# Parallel NeoOLAF documents. The streaming scheduler holds no more than this many jobs.
DOCUMENT_WORKERS = 4

# Existing v5.1 parallel relation-enrichment workers inside each document.
LAYER_WORKERS = 16

REASONING_EFFORT = "minimal"
MAX_TOKENS = 4096
REQUEST_TIMEOUT = 120

RESUME_COMPLETED = True
RETRY_FAILED_DOCUMENTS = True
DOCUMENT_ATTEMPTS = 2
RETRY_BACKOFF_SECONDS = 8.0
DOCUMENT_LAUNCH_STAGGER_SECONDS = 0.75
VERBOSE_DOCUMENTS = False
PROGRESS_EVERY = 1 if not RUN_ALL_DEV_DOCUMENTS else 10

print("Document workers / maximum pending documents:", DOCUMENT_WORKERS)
print("Layer workers per document:", LAYER_WORKERS)
print("Maximum theoretical concurrent Layer 2 requests:", DOCUMENT_WORKERS * LAYER_WORKERS)
print("API key available:", bool(API_KEY))

Document workers / maximum pending documents: 4
Layer workers per document: 16
Maximum theoretical concurrent Layer 2 requests: 64
API key available: True


## 3. Streaming preflight and anti-leakage assertions

In [4]:
required = [
    DATASET_JSONL,
    ONTOLOGY_PATH,
    ONTOLOGY_ORIGINAL,
    RELATION_CATALOG,
    RELATION_ALIASES,
    PROFILE_PATH,
    GUIDANCE_PATH,
    TASK_GUIDANCE_PATH,
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

profile = read_json(PROFILE_PATH)
task_guidance = read_json(TASK_GUIDANCE_PATH)
catalog = read_json(RELATION_CATALOG)

# One streaming scan: counts exact type="dev" records without retaining documents.
corpus_scan = count_exact_type_records(
    DATASET_JSONL,
    required_type=EXACT_TYPE_VALUE,
    first_n_ids=5,
)
matching_records = int(corpus_scan["matching_records"])
available_after_start = max(0, matching_records - START_DEV_INDEX)
selected_count = (
    available_after_start
    if RUN_ALL_DEV_DOCUMENTS
    else min(SMOKE_DEV_DOCUMENT_LIMIT, available_after_start)
)

print("JSONL records:", corpus_scan["total_records"])
print('Records with exact key/value type="dev":', matching_records)
print("Selected this invocation:", selected_count)
print("First matching dev IDs:", corpus_scan["first_matching_ids"])
print("Ontology properties:", catalog["property_count"])
print("Allowed relation IDs:", len(task_guidance["allowed_relation_ids"]))
print("Frozen profile:", profile["profile_name"])

assert EXACT_TYPE_VALUE == "dev"
assert catalog["property_count"] == 96
assert len(task_guidance["allowed_relation_ids"]) == 96
assert profile["relations"]["allowed"] == []
assert profile["anti_cheating"]["direct_docred_extraction"] is False
assert profile["anti_cheating"]["source_entity_anchoring"] is False
assert profile["anti_cheating"]["post_run_relation_invention"] is False
assert profile["benchmark_projection"]["gold_available_to_pipeline"] is False
assert 3 <= DOCUMENT_WORKERS <= 5
assert selected_count > 0

display(Markdown(
    f"**Mode:** {'all exact type=dev documents' if RUN_ALL_DEV_DOCUMENTS else 'first five exact type=dev documents'}  \n"
    f"**Selected:** {selected_count} of {matching_records} dev documents  \n"
    f"**Parent memory bound:** at most {DOCUMENT_WORKERS} submitted document jobs  \n"
    f"**Resume:** {RESUME_COMPLETED}; keep the same batch root when switching to all dev documents."
))

JSONL records: 106924
Records with exact key/value type="dev": 998
Selected this invocation: 998
First matching dev IDs: ['DocRED - e37288ca6012859f', 'DocRED - 72971f0f36693b3a', 'DocRED - 0af496a208c087f2', 'DocRED - 3b611c48af52425e', 'DocRED - 2d3e5aaa2e9e3e9b']
Ontology properties: 96
Allowed relation IDs: 96
Frozen profile: docred_native_country_compact_ablation_v5


**Mode:** all exact type=dev documents  
**Selected:** 998 of 998 dev documents  
**Parent memory bound:** at most 4 submitted document jobs  
**Resume:** True; keep the same batch root when switching to all dev documents.

## 4. Run native NeoOLAF with bounded document streaming

The launcher reads the JSONL iteratively and advances to another `type="dev"` record only after a document-worker slot is available. It does not prepare all dev records or submit all futures in advance.

Each selected document still receives its complete Layer 0–12 run, logs, prompts, raw responses, ontology artifacts, strict relation evaluation, entity evaluation, and cumulative layer analysis.

In [5]:
RUN_PIPELINE = True

if RUN_PIPELINE:
    if not API_KEY:
        API_KEY = getpass("OpenRouter API key: ").strip().strip('"').strip("'")
    if not API_KEY:
        raise RuntimeError("No OpenRouter API key was provided.")

    config = BatchRunConfig(
        project_root=str(PROJECT_ROOT),
        ontology_path=str(ONTOLOGY_PATH),
        profile_path=str(PROFILE_PATH),
        guidance_path=str(GUIDANCE_PATH),
        relation_catalog_path=str(RELATION_CATALOG),
        relation_aliases_path=str(RELATION_ALIASES),
        model_name=MODEL_NAME,
        host=OPENROUTER_HOST,
        document_workers=DOCUMENT_WORKERS,
        layer_workers=LAYER_WORKERS,
        reasoning_effort=REASONING_EFFORT,
        max_tokens=MAX_TOKENS,
        request_timeout=REQUEST_TIMEOUT,
        resume_completed=RESUME_COMPLETED,
        retry_failed_documents=RETRY_FAILED_DOCUMENTS,
        document_attempts=DOCUMENT_ATTEMPTS,
        retry_backoff_seconds=RETRY_BACKOFF_SECONDS,
        launch_stagger_seconds=DOCUMENT_LAUNCH_STAGGER_SECONDS,
        verbose_documents=VERBOSE_DOCUMENTS,
        progress_every=PROGRESS_EVERY,
    )

    batch = run_batch_streaming(
        dataset_jsonl=DATASET_JSONL,
        task_guidance_path=TASK_GUIDANCE_PATH,
        batch_root=BATCH_ROOT,
        run_all_documents=RUN_ALL_DEV_DOCUMENTS,
        smoke_document_limit=SMOKE_DEV_DOCUMENT_LIMIT,
        start_index=START_DEV_INDEX,
        config=config,
        api_key=API_KEY,
        matching_record_count=matching_records,
        required_type=EXACT_TYPE_VALUE,
    )
    print("Dev-only bounded streaming execution and aggregate evaluation completed.")
else:
    batch = aggregate_batch_results_streaming(
        batch_root=BATCH_ROOT,
        relation_catalog_path=RELATION_CATALOG,
    )
    print("RUN_PIPELINE=False: aggregate evaluation rebuilt iteratively from saved document files.")

[1/998] skipped_completed: DocRED - e37288ca6012859f | pipeline=20.073s


/home/galencarmedeiro/git/postdoc/NeoOLAF/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
/home/galencarmedeiro/git/postdoc/NeoOLAF/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
/home/galencarmedeiro/git/postdoc/NeoOLAF/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
/home/galencarmedeiro/git/postdoc/NeoOLAF/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


[10/998] completed: DocRED - 870d7d51388dd9c2 | pipeline=19.937s
[20/998] completed: DocRED - 1fe77c1886347cbb | pipeline=22.689s
[30/998] completed: DocRED - 376edafcd90c34f5 | pipeline=9.889s
[40/998] completed: DocRED - 865910b4761a8421 | pipeline=33.804s
[50/998] completed: DocRED - b575ebc1a539a591 | pipeline=23.559s
[60/998] completed: DocRED - d19f7b9f0509c1d2 | pipeline=11.831s
[70/998] completed: DocRED - 17b763e5945e3e10 | pipeline=25.136s
[80/998] completed: DocRED - ce6fe4fce2c8e9e3 | pipeline=0.989s
[90/998] completed: DocRED - 0ded5069fa77eb8b | pipeline=19.099s
[100/998] completed: DocRED - a98bdb34b2d7dd03 | pipeline=8.01s
[110/998] completed: DocRED - 791a4f4a6ac77472 | pipeline=8.383s
[120/998] completed: DocRED - 4db636c8ac36eec5 | pipeline=19.319s
[130/998] completed: DocRED - 4fff93af17cde36a | pipeline=13.806s
[140/998] completed: DocRED - 07c5152a92b5e0bc | pipeline=9.571s
[150/998] completed: DocRED - bd9d38a525c44235 | pipeline=17.801s
[160/998] completed: DocR

## 5. Aggregate relation and entity evaluation

In [6]:
summary = batch["summary"]
display(pd.DataFrame([{
    "documents_requested": summary["documents_requested"],
    "documents_completed_or_resumed": summary["documents_completed_or_resumed"],
    "documents_failed": summary["documents_failed"],
    "relation_micro_precision": summary["micro_relation"]["precision"],
    "relation_micro_recall": summary["micro_relation"]["recall"],
    "relation_micro_f1": summary["micro_relation"]["f1"],
    "relation_macro_precision": summary["macro_relation"]["precision"],
    "relation_macro_recall": summary["macro_relation"]["recall"],
    "relation_macro_f1": summary["macro_relation"]["f1"],
    "entity_micro_precision": summary["micro_entity_inventory"]["precision"],
    "entity_micro_recall": summary["micro_entity_inventory"]["recall"],
    "entity_micro_f1": summary["micro_entity_inventory"]["f1"],
    "endpoint_micro_precision": summary["micro_relation_endpoint_inventory"]["precision"],
    "endpoint_micro_recall": summary["micro_relation_endpoint_inventory"]["recall"],
    "endpoint_micro_f1": summary["micro_relation_endpoint_inventory"]["f1"],
    "mean_pipeline_seconds": summary["mean_document_pipeline_seconds"],
    "median_pipeline_seconds": summary["median_document_pipeline_seconds"],
}]))

print("First-failure counts across completed documents:")
display(pd.DataFrame(
    sorted(summary["first_failure_counts"].items(), key=lambda item: (-item[1], item[0])),
    columns=["first_failure", "gold_relations"],
))

,documents_requested,documents_completed_or_resumed,documents_failed,relation_micro_precision,relation_micro_recall,relation_micro_f1,relation_macro_precision,relation_macro_recall,relation_macro_f1,entity_micro_precision,entity_micro_recall,entity_micro_f1,endpoint_micro_precision,endpoint_micro_recall,endpoint_micro_f1,mean_pipeline_seconds,median_pipeline_seconds
0,998,978,20,0.433521,0.088494,0.146984,0.203505,0.090555,0.110263,1.0,0.50169,0.668167,0.738287,0.215055,0.333086,21.146232,12.977


First-failure counts across completed documents:


,first_failure,gold_relations
0,layer01_endpoint_missing_after_projection,6547
1,layer01_relation_instance_missing,4018
2,survived_to_layer05,1045
3,layer02_wrong_or_missing_controlled_relation,530
4,layer04_endpoint_direction_or_type_validation,19


## 6. Per-document results (loaded from the streamed CSV)

In [7]:
per_document_path = Path(batch["paths"]["per_document_csv"])
per_document = pd.read_csv(per_document_path) if per_document_path.is_file() else pd.DataFrame()

if not per_document.empty:
    print("Completed documents in CSV:", len(per_document))
    print("Twenty-five lowest relation-F1 documents:")
    display(per_document.sort_values(
        ["relation_f1", "relation_recall", "relation_precision"],
        ascending=[True, True, True],
    ).head(25))
    display(per_document[[
        "relation_precision", "relation_recall", "relation_f1",
        "entity_precision", "entity_recall", "entity_f1",
        "pipeline_seconds",
    ]].describe())
else:
    print("No completed documents.")

Completed documents in CSV: 978
Twenty-five lowest relation-F1 documents:


,selection_index,source_index,document_id,title,status,pipeline_seconds,wall_seconds,relation_predicted,relation_gold,relation_tp,...,relation_precision,relation_recall,relation_f1,entity_precision,entity_recall,entity_f1,endpoint_precision,endpoint_recall,endpoint_f1,run_dir
6,6,6,DocRED - 870d7d51388dd9c2,Ned McEvoy,completed,19.937,20.631,0,25,0,...,0.0,0.0,0.0,1.0,0.200000,0.333333,0.000000,0.000000,0.000000,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
9,9,9,DocRED - c7e20c1cd39660dc,Robert K. Huntington,completed,10.825,11.422,3,14,0,...,0.0,0.0,0.0,1.0,0.166667,0.285714,0.666667,0.166667,0.266667,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
10,10,10,DocRED - b7c1751749fd680d,More (The Sisters of Mercy song),completed,107.320,108.363,1,9,0,...,0.0,0.0,0.0,1.0,0.761905,0.864865,1.000000,0.200000,0.333333,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
11,11,11,DocRED - 6965789a3c1dcd11,Delaware General Assembly,completed,523.340,524.077,5,17,0,...,0.0,0.0,0.0,1.0,0.333333,0.500000,1.000000,0.750000,0.857143,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
13,13,13,DocRED - 410002886f5aaadc,Palestinian National Theatre,completed,5.981,6.599,0,3,0,...,0.0,0.0,0.0,1.0,0.846154,0.916667,0.000000,0.000000,0.000000,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
15,15,15,DocRED - 66489d4a482b1768,Allen F. Moore,completed,10.429,11.055,0,30,0,...,0.0,0.0,0.0,1.0,0.387097,0.558140,0.000000,0.000000,0.000000,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
16,16,16,DocRED - ac1cb5f651a13eb0,Johan Gottlieb Gahn,completed,23.007,23.760,0,7,0,...,0.0,0.0,0.0,1.0,0.500000,0.666667,0.000000,0.000000,0.000000,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
17,17,17,DocRED - 73adc17de8216965,Siberia Governorate,completed,12.640,13.270,1,11,0,...,0.0,0.0,0.0,1.0,0.333333,0.500000,0.500000,0.142857,0.222222,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
18,18,18,DocRED - 0446f49b345ba71c,Quokka,completed,25.655,26.201,0,5,0,...,0.0,0.0,0.0,1.0,0.857143,0.923077,0.000000,0.000000,0.000000,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
20,20,20,DocRED - ad28bc309b708830,Memogate (Pakistan),completed,4.853,5.355,0,9,0,...,0.0,0.0,0.0,1.0,0.095238,0.173913,0.000000,0.000000,0.000000,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...


,relation_precision,relation_recall,relation_f1,entity_precision,entity_recall,entity_f1,pipeline_seconds
count,978.000000,978.000000,978.000000,978.000000,978.000000,978.000000,978.000000
mean,0.203505,0.090555,0.110263,0.972393,0.517329,0.647758,21.146232
std,0.320054,0.165373,0.182704,0.163929,0.232298,0.227416,55.549520
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.537000
25%,0.000000,0.000000,0.000000,1.000000,0.363636,0.533333,6.027500
50%,0.000000,0.000000,0.000000,1.000000,0.533333,0.695652,12.977000
75%,0.375000,0.125000,0.181818,1.000000,0.692308,0.818182,20.896500
max,1.000000,1.000000,0.937500,1.000000,1.000000,1.000000,858.986000


## 7. Per-relation micro evaluation

In [8]:
per_relation = pd.DataFrame(batch["per_relation"])
if not per_relation.empty:
    display(per_relation.sort_values(
        ["gold", "f1", "recall"],
        ascending=[False, True, True],
    ))
else:
    print("No relation metrics available.")

,relation_id,label,predicted,gold,true_positive,false_positive,false_negative,precision,recall,f1
27,P17,,1332,2779,777,555,2002,0.583333,0.279597,0.378010
9,P131,,275,1226,92,183,1134,0.334545,0.075041,0.122585
46,P27,,55,804,37,18,767,0.672727,0.046020,0.086147
20,P150,,0,608,0,0,608,0.000000,0.000000,0.000000
76,P577,,48,405,34,14,371,0.708333,0.083951,0.150110
...,...,...,...,...,...,...,...,...,...,...
84,P676,,0,8,0,0,8,0.000000,0.000000,0.000000
70,P551,,1,6,0,1,6,0.000000,0.000000,0.000000
92,P807,,7,2,0,7,2,0.000000,0.000000,0.000000
36,P190,,1,2,1,0,1,1.000000,0.500000,0.666667


## 8. Aggregate cumulative layer contribution

In [9]:
cumulative = pd.DataFrame(batch["cumulative"])
if not cumulative.empty:
    display(cumulative[[
        "layer_index", "layer_name", "predicted", "gold",
        "true_positive", "false_positive", "false_negative",
        "precision", "recall", "f1",
    ]])
else:
    print("No cumulative layer metrics available.")

,layer_index,layer_name,predicted,gold,true_positive,false_positive,false_negative,precision,recall,f1
0,0,layer00_preprocessing,0,12159,0,0,12159,0.000000,0.000000,0.000000
1,1,layer01_linguistic_expression_extraction,0,12159,0,0,12159,0.000000,0.000000,0.000000
2,2,layer02_candidate_enrichment,0,12159,0,0,12159,0.000000,0.000000,0.000000
3,3,layer03_candidate_typing_resolution,0,12159,0,0,12159,0.000000,0.000000,0.000000
4,4,layer04_candidate_relation_extraction,2482,12159,1076,1406,11083,0.433521,0.088494,0.146984
5,5,layer05_candidate_triple_generation,2482,12159,1076,1406,11083,0.433521,0.088494,0.146984
6,6,layer06_concept_relation_induction,2482,12159,1076,1406,11083,0.433521,0.088494,0.146984
7,7,layer07_hierarchisation,2482,12159,1076,1406,11083,0.433521,0.088494,0.146984
8,8,layer08_axiom_schemata_extraction,2482,12159,1076,1406,11083,0.433521,0.088494,0.146984
9,9,layer09_general_axiom_extraction,2482,12159,1076,1406,11083,0.433521,0.088494,0.146984


## 9. Failures, scheduler state, and saved outputs

In [10]:
failed = pd.DataFrame(batch.get("failed_preview", []))
print("Failed documents:", summary["documents_failed"])
if not failed.empty:
    display(failed[[
        column for column in [
            "selection_index", "document_id", "title", "status",
            "error_type", "error", "run_dir",
        ]
        if column in failed.columns
    ]])

if "scheduler" in batch:
    display(pd.DataFrame([batch["scheduler"]]))

print("Batch output root:", BATCH_ROOT)
print("Selection stream:", BATCH_ROOT / "selection_documents.jsonl")
print("Selection manifest:", BATCH_ROOT / "selection_manifest.json")
print("Progress events:", BATCH_ROOT / "batch_events.jsonl")
print("Aggregate summary:", batch["paths"]["batch_summary"])
print("Per-document CSV:", batch["paths"]["per_document_csv"])
print("Per-relation CSV:", BATCH_ROOT / "aggregate_analysis/per_relation_metrics.csv")
print("Cumulative layer CSV:", BATCH_ROOT / "aggregate_analysis/cumulative_layer_micro_evaluation.csv")
print("Predictions JSONL:", batch["paths"]["predictions_jsonl"])
print("Failures JSONL:", batch["paths"]["failed_documents_jsonl"])

Failed documents: 20


,selection_index,document_id,title,status,error_type,error,run_dir
0,180,DocRED - 41d762b7738a6d38,M. C. Veerabahu Pillai,failed,RuntimeError,No final message.content returned by backend o...,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
1,386,DocRED - d5afb4f79b8f78bf,Economic history of Cambodia,failed,IndexError,list index out of range,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
2,415,DocRED - cf134da0ca9d513b,Vesper sparrow,failed,IndexError,list index out of range,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
3,418,DocRED - f1c3a9b6567acdea,Gromshin Heights,failed,IndexError,list index out of range,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
4,458,DocRED - 2e2db7fa57b135a2,"Afonso, Prince Imperial of Brazil",failed,RuntimeError,layer01_country_aware: response length 29639 e...,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
5,482,DocRED - aeb35b88a907b67f,Ages of consent in Oceania,failed,IndexError,list index out of range,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
6,508,DocRED - 63e458d75bdcdc0b,Jane Livingston,failed,IndexError,list index out of range,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
7,523,DocRED - d496f3bf42d98809,Wayne Gretzky Drive,failed,RuntimeError,No final message.content returned by backend o...,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
8,529,DocRED - 753581dbb1e75a38,Uptoi Village,failed,IndexError,list index out of range,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
9,537,DocRED - 34443841a446b053,Phyllonorycter issikii,failed,IndexError,list index out of range,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...


,expected_total,documents_prepared,documents_finished,documents_completed_or_resumed,documents_failed,documents_resumed,max_pending_documents,elapsed_seconds
0,998,998,998,978,20,5,4,5791.882


Batch output root: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/docred_native_v5_1_dev_streaming
Selection stream: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/docred_native_v5_1_dev_streaming/selection_documents.jsonl
Selection manifest: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/docred_native_v5_1_dev_streaming/selection_manifest.json
Progress events: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/docred_native_v5_1_dev_streaming/batch_events.jsonl
Aggregate summary: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/docred_native_v5_1_dev_streaming/aggregate_analysis/batch_summary.json
Per-document CSV: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/docred_native_v5_1_dev_streaming/aggregate_analysis/per_document_metrics.csv
Per-relation CSV: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/docred_native_v5_1

## 10. Launch the complete dev split

After the five-document dev smoke result is acceptable, change only:

```python
RUN_ALL_DEV_DOCUMENTS = True
```

Rerun the switch/configuration, preflight, and batch cells. Keep the same `BATCH_ROOT`. Completed smoke documents are detected by their scientific fingerprints and skipped.

The full run still reads all 106,924 JSONL lines to locate matching records, but it retains only the current line and at most `DOCUMENT_WORKERS` submitted document jobs. It does **not** create a list of every dev document in RAM.

For sustained HTTP 429 errors, reduce `DOCUMENT_WORKERS` from `4` to `3` before reducing `LAYER_WORKERS`.